# 00 · Ingeniería de datos: unificación de los 6 CSV reales

**Proyecto:** predicción de deserción del tratamiento de fisura labiopalatina.

Este notebook documenta el pipeline ETL que consume los archivos **reales** de `data/raw/`
(no datos simulados). La implementación ejecutable está en
`notebooks/00_procesamiento_dataset_unificado.py`.

## Objetivo
Construir `data/processed/dataset_final_limpio.csv` con **una fila por paciente**,
etiqueta `desertor` y variables clínicas, quirúrgicas, de terapia y socioeconómicas
alineadas entre las cohortes v1 (PAC-1000) y v2 (PAC-2000).

## Fuentes
| Archivo | Rol | Llave |
|---|---|---|
| `dataset_desercion_fisura_bilabial.csv` | Deserción y contexto v1 | `ID_Paciente` |
| `dataset_hitos_quirurgicos_complicaciones.csv` | Cirugía v1 | `ID_Paciente` |
| `dataset_terapia_lenguaje_ivf.csv` | Terapia / IVF v1 | `ID_Paciente` |
| `dataset_v2_desercion_socioeconomica.csv` (alias: `dataset_desercion_socioeconomica.csv`) | Deserción v2 | `ID_Paciente` |
| `dataset_v2_hitos_quirurgicos.csv` | Cirugía v2 | `ID_Paciente` |
| `dataset_v2_terapia_lenguaje.csv` | Terapia v2 | `ID_Paciente` |

## Criterios de merge y nulos
- Join **left** desde las tablas con `Desertor`: sin etiqueta no hay caso de modelado.
- Pacientes v1 con cirugía/terapia pero **sin** ficha de deserción se excluyen (no se imputa el target).
- Numéricas: imputación por **mediana de cohorte** (evita mezclar escalas v1/v2).
- `Cirugias_Programadas` no existe en v2: se estima con 2 hitos protocolarios + complicaciones.
- Categóricas: `tipo_fisura` armonizado (se retira el prefijo "Bilateral") y SES ordinal 0–4.


In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 110,
})

CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next(
    path for path in CANDIDATES
    if (path / "data" / "processed" / "dataset_final_limpio.csv").exists()
    or (path / "data" / "raw" / "dataset_desercion_fisura_bilabial.csv").exists()
)
PROCESSED = ROOT / "data" / "processed" / "dataset_final_limpio.csv"
RAW = ROOT / "data" / "raw"
print(f"Raíz del proyecto: {ROOT}")


Raíz del proyecto: C:\Users\richa\Downloads\Proyecto-predicci-n-de-desertamiento-del-tratamiento-de-fisura-bilabial


In [2]:
from importlib.machinery import SourceFileLoader

etl = SourceFileLoader(
    "etl_fisura",
    str(ROOT / "notebooks" / "00_procesamiento_dataset_unificado.py"),
).load_module()

inventario = []
for clave, archivo in etl.RAW_FILES.items():
    path = etl.resolve_raw_path(archivo)
    tmp = pd.read_csv(path)
    inventario.append({
        "clave": clave,
        "archivo": path.name,
        "filas": len(tmp),
        "columnas": tmp.shape[1],
        "ids_unicos": tmp["ID_Paciente"].nunique(),
        "tiene_desertor": "Desertor" in tmp.columns,
    })
inventario_df = pd.DataFrame(inventario)
inventario_df


,clave,archivo,filas,columnas,ids_unicos,tiene_desertor
0,desercion_v1,dataset_desercion_fisura_bilabial.csv,500,11,500,True
1,cirugia_v1,dataset_hitos_quirurgicos_complicaciones.csv,1000,7,1000,False
2,terapia_v1,dataset_terapia_lenguaje_ivf.csv,1000,6,1000,False
3,desercion_v2,dataset_v2_desercion_socioeconomica.csv,3000,9,3000,True
4,cirugia_v2,dataset_v2_hitos_quirurgicos.csv,3000,7,3000,False
5,terapia_v2,dataset_v2_terapia_lenguaje.csv,3000,6,3000,False


## Inventario y calidad de llaves

Se verifica que cada CSV tenga `ID_Paciente`, que no haya IDs vacíos y que el merge
v1/v2 no duplique pacientes entre cohortes.


In [3]:
dataset = etl.build_unified_dataset()
output = etl.save_processed(dataset)
print(f"Guardado en: {output}")
print(f"Forma final: {dataset.shape}")
print("Nulos por columna:")
print(dataset.isna().sum().sort_values(ascending=False).head(10))
print("\nCohortes:", dataset["cohorte"].value_counts().to_dict())
print("Tasa de deserción global:", round(float(dataset["desertor"].mean()), 4))
print("Tasa por cohorte:")
print(dataset.groupby("cohorte")["desertor"].mean())
dataset.head()


Guardado en: C:\Users\richa\Downloads\Proyecto-predicci-n-de-desertamiento-del-tratamiento-de-fisura-bilabial\data\processed\dataset_final_limpio.csv
Forma final: (3500, 31)
Nulos por columna:
id_paciente                 0
cohorte                     0
tipo_fisura                 0
fisura_Completa             0
fisura_Asimetrica           0
fisura_Incompleta           0
edad_inicio_meses           0
distancia_hospital_km       0
nivel_socioeconomico        0
nivel_socioeconomico_ord    0
dtype: int64

Cohortes: {'v2': 3000, 'v1': 500}
Tasa de deserción global: 0.7463
Tasa por cohorte:
cohorte
v1    0.940
v2    0.714
Name: desertor, dtype: float64


,id_paciente,cohorte,tipo_fisura,fisura_Completa,fisura_Asimetrica,fisura_Incompleta,edad_inicio_meses,distancia_hospital_km,nivel_socioeconomico,nivel_socioeconomico_ord,...,fistula_palatina_residual,complicaciones_quirurgicas,peso_z_score_2anos,citas_terapia_programadas,citas_terapia_asistidas,ratio_asistencia_terapia,hipoacusia_conductiva,grado_hipernasalidad,insuficiencia_velofaringea_ivf,desertor
0,PAC-1000,v1,Completa,1,0,0,6,46.9,Alto,4,...,1,1,0.16,20,15,0.750000,0,2,1,1
1,PAC-1001,v1,Asimetrica,0,1,0,1,59.3,Bajo,0,...,0,0,-0.37,47,46,0.978723,0,3,1,1
2,PAC-1002,v1,Incompleta,0,0,1,9,9.2,Medio,2,...,0,0,-0.25,43,31,0.720930,1,0,0,1
3,PAC-1003,v1,Incompleta,0,0,1,1,88.2,Medio,2,...,0,0,-1.07,22,14,0.636364,0,2,1,1
4,PAC-1004,v1,Completa,1,0,0,5,19.1,Bajo,0,...,0,0,-2.21,35,33,0.942857,1,3,1,1


## Conclusiones del procesamiento
- El dataset analítico es el producto de **6 archivos reales**, no de un mock.
- La unidad de análisis es el paciente (`id_paciente`).
- Las variables están en `snake_case`, con dummies de fisura y SES ordinal listos para XGBoost.
- El desbalance de `desertor` es estructural (más marcado en v1) y debe tratarse con `scale_pos_weight` en modelado, no borrando casos.
